## data preparation

In [ ]:
import os
import json
import requests
from tqdm import tqdm  # 导入进度条库

def download_files(url, path):
    # 确保目标文件夹存在
    directory = os.path.dirname(path)
    if directory:
        os.makedirs(directory, exist_ok=True)

    # 1. 下载阶段（带进度条）
    try:
        print(f"准备连接并下载至 {path}...")
        response = requests.get(url, stream=True, timeout=15)
        response.raise_for_status()

        # 获取文件的总大小（单位：字节）
        total_size = int(response.headers.get('content-length', 0))
        chunk_size = 8192

        # 初始化 tqdm 进度条
        progress_bar = tqdm(total=total_size, unit='iB', unit_scale=True, desc="下载进度")

        with open(path, mode='wb') as file:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    file.write(chunk)
                    progress_bar.update(len(chunk)) # 更新进度条

        progress_bar.close()

        # 检查是否下载完整
        if total_size != 0 and progress_bar.n != total_size:
            print("警告：下载可能不完整，请检查网络！")
            return
        print("\n下载完成！")

    except Exception as error:
        print(f"\n下载失败: {error}")
        return

    # 2. 解析阶段
    try:
        print("正在将大文件加载到内存并解析 JSON（这可能需要一些时间，请耐心等待）...")
        with open(path, mode='r', encoding='utf-8') as f:
            data = json.load(f)
            print("解析成功！数据前 2 个元素如下：")

            if isinstance(data, list):
                print(data[:2]) # 数据太大，先打印 2 个看看
            elif isinstance(data, dict):
                print(list(data.items())[:2])

    except json.JSONDecodeError:
        print("JSON 解析失败！原因极有可能是该文件为 JSONL 格式（每行是一个独立的JSON）。")
        print("如果是 JSONL 格式，请不要使用 json.load()，需要按行读取。")
    except Exception as error:
        print(f"读取文件时发生错误: {error}")

url = "https://huggingface.co/datasets/zxbsmk/webnovel_cn/resolve/main/novel_cn_token512_50k.json?download=true"
save_path = "data/scifi-finetune.json"

download_files(url=url, path=save_path)



In [ ]:
import glob
import os
path = './data'
story = '科幻小说'
def find_files(path, story: str):
    return glob.glob(os.path.join(path, story, '**', '*.txt'), recursive=True)


files = find_files(path=path, story=story)
def concatenate_txt_files(files):
    with open('./data/scifi.txt', mode = 'w') as outfile:
        for file in files:
            with open(file=file, mode='r') as f:
                content = f.read()
            outfile.write(content + '\n')
concatenate_txt_files(files=files)

## model

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

context_length = 128
d_model = 512
num_heads = 8
num_blocks = 12
dropout = 0.1
device = 'cuda' if torch.cuda.is_available() else 'cpu'
seed = 1
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.d_model = d_model
        self.ffn = nn.Sequential(
            nn.Linear(self.d_model, 4*self.d_model),
            nn.ReLU(),
            nn.Linear(4*self.d_model, self.d_model),
            nn.Dropout(dropout)
        )
    def forward(self,x):
        return self.ffn(x)

class Attention(nn.Module):
    def __init__(self):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_size = self.d_model // self.num_heads
        self.context_length = context_length
        self.dropout = nn.Dropout(dropout)
        self.Wq = nn.Linear(self.d_model, self.head_size, bias= False)
        self.Wk = nn.Linear(self.d_model, self.head_size, bias= False)
        self.Wv = nn.Linear(self.d_model, self.head_size, bias= False)
        # 因果掩码：下三角矩阵，上三角部分填充 -inf
        self.register_buffer(name='tril', tensor=torch.tril(torch.ones(size=[self.context_length, self.context_length])))

    def forward(self, x):
        B, T, C = x.shape
        q = self.Wq(x)
        k = self.Wk(x)
        v = self.Wv(x)

        attention = q @ k.transpose(-1,-2) / math.sqrt(self.head_size)
        attention = attention.masked_fill(self.tril[:T,:T] == 0, float('-inf'))
        attention = F.softmax(attention, dim = -1)
        attention = self.dropout(attention) @ v
        return attention

class MultiheadAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.dropout = nn.Dropout(dropout)
        self.head_size = self.d_model // self.num_heads
        self.heads = nn.ModuleList([Attention() for _ in range(self.num_heads)])
        self.linear_output = nn.Linear(self.d_model, self.d_model)

    def forward(self, x):
        head_output = [h(x) for h in self.heads]
        output = torch.cat(head_output, dim=-1)
        output = self.linear_output(output)
        output = self.dropout(output)
        return output

class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.d_model = d_model
        self.ln1 = nn.LayerNorm(self.d_model)
        self.ln2 = nn.LayerNorm(self.d_model)
        self.ffn = FeedForward()
        self.multiheadattention = MultiheadAttention()

    def forward(self, x):
        x = x + self.multiheadattention(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class Model(nn.Module):
    def __init__(self,  max_token_value=100256):
        super().__init__()
        self.d_model = d_model
        self.context_length = context_length
        self.input_embedding_lookup_table = nn.Embedding(max_token_value, self.d_model)
        self.transformer_blocks = nn.Sequential(
            *([TransformerBlock() for _ in range(num_blocks)] + [nn.LayerNorm(self.d_model)] )
        )
        self.final_output = nn.Linear(self.d_model, max_token_value)
        # 预先计算位置编码查找表
        position_embedding_lookup = torch.ones((self.context_length, self.d_model))
        position = torch.arange(0, self.context_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(start=0, end=self.d_model, step=2).float() * (-math.log(10000) / self.d_model))
        position_embedding_lookup[:, 0::2] = torch.sin(position * div_term)
        position_embedding_lookup[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('position_embedding_lookup', position_embedding_lookup)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        position_embedding = self.position_embedding_lookup[:T, :].to(idx.device)
        x = self.input_embedding_lookup_table(idx)
        x = x + position_embedding  # 位置编码加到输入上
        out = self.transformer_blocks(x)
        logits = self.final_output(out)
        if targets is not None:
            reshaped_logits = logits.reshape(B*T, -1)
            reshaped_labels = targets.reshape(B*T)
            loss = F.cross_entropy(reshaped_logits, reshaped_labels)
            return logits, loss
        else:
            loss = None
            return logits, loss

    def generate(self, idx, max_new_tokens=100):
        for _ in range(max_new_tokens):
            idx_crop = idx[:, -self.context_length:]
            logits, _ = self(idx_crop)
            logit = logits[:, -1, :]
            probs = F.softmax(logit, dim=-1)
            idx_next = torch.multinomial(input=probs, num_samples=1)
            idx = torch.cat([idx, idx_next], dim=1)
        return idx

## train


In [ ]:
import os
import sys
import pickle
from contextlib import nullcontext
import torch
import tiktoken
from aim import Run
# from model import Model


# Hyperparameters
batch_size = 12  # How many batches per training step
context_length = 128  # Length of the token chunk each batch
max_iters = 20000  # Total of training iterations <- Change this to smaller number for testing
learning_rate = 1e-3  # 0.001
eval_interval = 50  # How often to evaluate
eval_iters = 20  # Number of iterations to average for evaluation
device = 'cuda' if torch.cuda.is_available() else 'cpu'  # Use GPU if it's available.
TORCH_SEED = 1337
torch.manual_seed(TORCH_SEED)

# AIM Logs
run = Run()
run["hparams"] = {
    "learning_rate": learning_rate,
    "max_iters": max_iters,
    "batch_size": batch_size,
    "context_length": context_length
}

with open('./data/scifi.txt', mode='r', encoding='gbk', errors='ignore') as f:
    text = f.read()

encoding = tiktoken.get_encoding('cl100k_base')
tokenized_text = encoding.encode(text)
tokenized_text = torch.tensor(tokenized_text, dtype=torch.long).to(device)

index = int(len(tokenized_text) * 0.9)
train_data = tokenized_text[:index]
valid_data = tokenized_text[index:]

def get_batch(split: str):
    data = train_data if split == 'train' else valid_data
    idxs = torch.randint(low=0, high=len(data) - context_length, size=(batch_size,))
    x = torch.stack([data[idx: idx + context_length] for idx in idxs]).to(device)
    y = torch.stack([data[idx+1: idx+context_length + 1] for idx in idxs]).to(device)
    return x, y

model = Model().to(device)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for key in ['train', 'valid']:
        loss_eval = torch.zeros(size = (eval_iters,))
        for i in range(eval_iters):
            x_batch, y_batch = get_batch(key)
            _, loss = model(x_batch, y_batch)
            loss_eval[i] = loss.item()
        out[key] = loss_eval.mean()
    model.train()
    return out

# Create the optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
tracked_losses = list()
for step in range(max_iters):
    if step % eval_iters == 0 or step == max_iters - 1:
        losses = estimate_loss()
        tracked_losses.append(losses)
        print('Step:', step, 'Training Loss:', round(losses['train'].item(), 3), 'Validation Loss:', round(losses['valid'].item(), 3))
        run.track(round(losses['train'].item(), 3), name='Training Loss')
        run.track(round(losses['valid'].item(), 3), name='Validation Loss')

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


In [35]:
# Save the model
if not os.path.exists('model'):
    os.makedirs('model')
torch.save(model.state_dict(), 'model/model-scifi.pt')

In [38]:
# -*- coding: utf-8 -*-
"""
Sample from a trained model
"""
import os
import pickle
from contextlib import nullcontext
import torch
import tiktoken
# from model import Model


# Hyperparameters
device = 'cuda' if torch.cuda.is_available() else 'cpu'
TORCH_SEED = 1337
torch.manual_seed(TORCH_SEED)
torch.cuda.manual_seed(TORCH_SEED)


encoding = tiktoken.get_encoding("cl100k_base")


# Initiate from trained model
model = Model()
model.load_state_dict(torch.load('model/model-scifi.pt'))
model.eval()
model.to(device)

# start = 'Write a short story about Sam Altman.'
start = '你好'
start_ids = encoding.encode(start)
x = (torch.tensor(start_ids, dtype=torch.long, device=device)[None, ...])

# run generation
with torch.no_grad():
    y = model.generate(x, max_new_tokens=500)
    print('---------------')
    print(encoding.decode(y[0].tolist()))
    print('---------------')

---------------
你好�
---------------


In [ ]:
### finetune
import torch
import tiktoken
import json
device = 'cuda' if torch.cuda.is_available() else 'cpu'
TORCH_SEED = 1337
torch.manual_seed(TORCH_SEED)
torch.cuda.manual_seed(TORCH_SEED)

# Hyperparameters (与训练时保持一致)
batch_size = 12
context_length = 128
max_iters = 100
learning_rate = 1e-3
eval_iters = 20

with open('data/scifi-finetune.json', mode='r', encoding='utf-8') as f:
    finetune_text = json.load(f)
    text = finetune_text[1000:5001]
# print(text)
encoding = tiktoken.get_encoding('cl100k_base')
tokenized_text = encoding.encode(str(text))

tokenized_text = torch.tensor(tokenized_text, dtype=torch.long)

train_index = int(len(tokenized_text) * 0.8)
train_data = tokenized_text[:train_index]
valid_data = tokenized_text[train_index:]

model = Model().to(device)

# get batch
def get_batch(split: str):
    data = train_data if split == 'train' else valid_data
    idxs = torch.randint(low=0, high=len(data) - context_length, size=(batch_size,))
    x = torch.stack([data[idx:idx + context_length] for idx in idxs]).to(device)
    y = torch.stack([data[idx + 1:idx + context_length + 1] for idx in idxs]).to(device)
    return x, y


# calculate the loss
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'valid']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x_batch, y_batch = get_batch(split)
            logits, loss = model(x_batch, y_batch)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


# Create the optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
tracked_losses = list()
for step in range(max_iters):
    if step % eval_iters == 0 or step == max_iters - 1:
        losses = estimate_loss()
        tracked_losses.append(losses)
        print('Step:', step, 'Training Loss:', round(losses['train'].item(), 3), 'Validation Loss:', round(losses['valid'].item(), 3))

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# Save the model
torch.save(model.state_dict(), 'model/model-scifi-finetune.pt')